# otro_pipe — un solo notebook, cliente x producto, grilla resumible

Reset de pipeline pedido despues de saltar entre `pipe_nuevo`/`residuos_2`/
`pipe_demanda`: preprocesamiento (cartesiano, TODOS los productos por
defecto) + FE + una grilla de 20 combos (escalado x variable objetivo),
cada uno con su propio Optuna, resumible si se corta a mitad de camino.

La mayor parte de las piezas de preprocesamiento/FE ya existen, probadas,
en `pipe_nuevo/01_Preprocesamiento.ipynb` y `pipe_nuevo/02_FE.ipynb` — se
portan tal cual, solo cambiando el default de `solo_productos_target` a
`False`. Lo nuevo: racha consecutiva, escalados `rango`/`normalpower`
(IQR), targets `log_ton_norm`/`delta_mean_12`/`delta_reg`, filtro de
universo de entrenamiento (80% de demanda + productos magicos) con
fallback para el resto, y el motor de grilla con resume.


## 0) Setup


In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import lightgbm as lgb
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


def _leer_json_reintentando(path, intentos=5, espera=2):
    """El bucket es un mount GCS FUSE: a veces tira Input/output error transitorio."""
    ultimo_error = None
    for _ in range(intentos):
        try:
            with open(path, encoding="utf-8") as f:
                return json.load(f)
        except OSError as e:
            ultimo_error = e
            time.sleep(espera)
    raise ultimo_error


BUCKET    = resolver_bucket()
DIR_RAW   = BUCKET / "datasets"
DIR_PRE   = BUCKET / "datasets_pre"      # cache del cartesiano (paso 1)
DIR_FE    = BUCKET / "datasets_fe"       # cache de FE (paso 2) + productos_magicos.json
RUTA_EXP  = BUCKET / "exp_otro_pipe"
for d in (DIR_PRE, DIR_FE, RUTA_EXP):
    d.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"cache pre: {DIR_PRE}")
print(f"cache FE : {DIR_FE}")
print(f"salida   : {RUTA_EXP}")


def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


In [ ]:
PARAM = {
    # ── Preprocesamiento (cartesiano cliente x producto x periodo) ────────
    # Default False: TODOS los productos, no solo los del target -- pedido
    # explicito ("dejar activado todos los productos"). El filtro a lo que
    # realmente importa se hace despues (80% de demanda + magicos), no aca.
    'solo_productos_target': False,
    'horizonte': 2,

    # ── FE ──────────────────────────────────────────────────────────────
    'max_lags': 12,
    'ventanas_ma': (3, 6, 12),          # se agrega 12 (antes solo 3 y 6)
    'ventanas_racha': (3, 6),           # NUEVO: racha consecutiva
    'niveles_share': ('cat1', 'cat2', 'cat3', 'mercado'),
    'lags_share': 3,
    'techo_indice': 10.0,
    'mes_corte_vecinos': 201905,
    'n_vecinos': 3,

    # ── Particion train/val/test ───────────────────────────────────────
    # meses_val ensanchado de 2 a 6 (201903-201908): con solo 2 meses, Optuna
    # elegia hiperparametros ajustando ruido puntual de esos meses -- se vio
    # como brechas val->test enormes y erraticas entre combos. TRAIN se acorta
    # 4 meses (pierde 201902-201905) para abrirle lugar a VAL sin tocar el gap
    # de leakage (siempre >= horizonte) ni el inicio de TEST.
    'meses_train': rango_meses(201701, 201901),
    'meses_val':   rango_meses(201903, 201908),
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── Filtro del universo de entrenamiento sofisticado ──────────────────
    'pct_demanda_cubrir': 0.80,
    'archivo_productos_magicos': 'productos_magicos.json',

    # ── La grilla: 6 escalados x 3 targets escalado-nativos + 2 targets
    #    fijos (delta_mean_12, delta_reg no varian por escalado -- ver
    #    plan: LightGBM es invariante a reescalados afines de los inputs,
    #    asi que repetirlos por escalado no cambiaria nada) ────────────────
    'escalados': ('mediana', 'media', 'zscore', 'rolling_mean', 'rango', 'normalpower'),
    'ventana_escalado_rolling': 3,
    'targets_escalado_nativos': ('ton_norm', 'delta_ton_norm', 'log_ton_norm'),
    'targets_fijos': ('delta_mean_12', 'delta_reg'),
    'salto_delta': None,     # None -> usa 'horizonte'
    'ridge_alpha': 1.0,

    # ── Optuna por combo ────────────────────────────────────────────────
    'n_trials': 30,
    'backup_cada_n_trials': 10,
    'regularizacion': 'normal',
    'techo_arboles': 500,
    'early_stopping_rounds': 50,

    # ── Entrega ─────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semilla_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'pausa_entre_submits_seg': 2,

    'semilla': 102191,
    'sufijo': '',
}
if PARAM['salto_delta'] is None:
    PARAM['salto_delta'] = PARAM['horizonte']

H = PARAM['horizonte']
L = PARAM['max_lags']
KEYS = ['product_id', 'customer_id']
KEYS_SQL = ", ".join(KEYS)

GRILLA = ([(t, e) for t in PARAM['targets_escalado_nativos'] for e in PARAM['escalados']]
         + [(t, None) for t in PARAM['targets_fijos']])
print(f"grilla: {len(GRILLA)} combos "
     f"({len(PARAM['targets_escalado_nativos'])} targets x {len(PARAM['escalados'])} escalados "
     f"+ {len(PARAM['targets_fijos'])} targets fijos)")
for t, e in GRILLA:
    print(f"  - target={t:16s} escalado={e}")


## 1) Preprocesamiento — cartesiano cliente x producto x periodo

Portado de `pipe_nuevo/01_Preprocesamiento.ipynb` (a su vez, el mismo
cartesiano de `src/workflow/z601_workflow_ceros.ipynb`): relleno de ceros
acotado por la vida del PRODUCTO y la fecha de alta del CLIENTE. Cacheado
en disco — si el parquet ya existe, se lee directo (para poder retomar sin
recalcular esto de nuevo).


In [ ]:
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE_PRE = f"sellin_zeroes_grpClienteProducto{_tgt}_otroPipe.parquet"
path_pre = DIR_PRE / NOMBRE_PRE

if path_pre.exists():
    print(f"cache encontrada: {path_pre.name} -> se salta el preprocesamiento")
else:
    t0 = time.time()
    con = duckdb.connect()

    con.execute(f"""
        CREATE OR REPLACE TABLE tb_sellin AS
        SELECT customer_id, product_id, periodo, plan_precios_cuidados,
               cust_request_qty, cust_request_tn, tn
        FROM read_csv_auto('{DIR_RAW / "sell-in.txt.gz"}')
        ORDER BY customer_id, product_id, periodo
    """)

    if PARAM['solo_productos_target']:
        con.execute(f"""
            CREATE OR REPLACE TABLE tb_target AS
            SELECT DISTINCT product_id
            FROM read_csv_auto('{DIR_RAW / "product_id_apredecir201912.txt"}')
        """)
        antes = con.sql("SELECT COUNT(DISTINCT product_id) FROM tb_sellin").fetchone()[0]
        con.execute("""
            CREATE OR REPLACE TABLE tb_sellin AS
            SELECT s.* FROM tb_sellin s
            WHERE s.product_id IN (SELECT product_id FROM tb_target)
        """)
        despues = con.sql("SELECT COUNT(DISTINCT product_id) FROM tb_sellin").fetchone()[0]
        print(f"solo productos target: {antes} -> {despues} productos")
    else:
        print("solo_productos_target=False -> TODOS los productos activados")

    n_sellin = con.sql("SELECT COUNT(*) FROM tb_sellin").fetchone()[0]
    print(f"tb_sellin: {n_sellin:,} filas   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tb_productos_fechas AS
        SELECT product_id, MIN(periodo) AS periodo_min, MAX(periodo) AS periodo_max
        FROM tb_sellin GROUP BY product_id
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tb_clientes_fechas AS
        SELECT customer_id, MIN(periodo) AS periodo_min
        FROM tb_sellin GROUP BY customer_id
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tb_periodos AS
        SELECT DISTINCT periodo FROM tb_sellin ORDER BY 1
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tb_precios_cuidados AS
        SELECT product_id, MIN(periodo) AS periodo_min, MAX(periodo) AS periodo_max
        FROM tb_sellin WHERE plan_precios_cuidados = 1
        GROUP BY product_id ORDER BY 1
    """)
    print(f"periodos : {con.sql('SELECT COUNT(*) FROM tb_periodos').fetchone()[0]}")
    print(f"productos: {con.sql('SELECT COUNT(*) FROM tb_productos_fechas').fetchone()[0]}")
    print(f"clientes : {con.sql('SELECT COUNT(*) FROM tb_clientes_fechas').fetchone()[0]}")
    print(f"[{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tb_zeroes AS
        SELECT cf.customer_id, pf.product_id, per.periodo,
               CAST(0 AS INT) AS plan_precios_cuidados,
               CAST(0 AS INT) AS cust_request_qty,
               0.0 AS cust_request_tn,
               0.0 AS tn
        FROM tb_productos_fechas pf, tb_clientes_fechas cf, tb_periodos per
        WHERE NOT EXISTS (
                SELECT 1 FROM tb_sellin si
                WHERE si.periodo = per.periodo
                  AND si.customer_id = cf.customer_id
                  AND si.product_id = pf.product_id
              )
          AND per.periodo BETWEEN pf.periodo_min AND pf.periodo_max
          AND per.periodo >= cf.periodo_min
        ORDER BY 1, 2, 3
    """)
    con.execute("""
        UPDATE tb_zeroes z
        SET plan_precios_cuidados = 1
        WHERE EXISTS (SELECT 1 FROM tb_precios_cuidados p
                      WHERE z.periodo BETWEEN p.periodo_min AND p.periodo_max
                        AND p.product_id = z.product_id)
    """)
    n_zeroes = con.sql("SELECT COUNT(*) FROM tb_zeroes").fetchone()[0]
    print(f"filas de cero agregadas: {n_zeroes:,}   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tb_sellin_zeroes AS
        SELECT * FROM tb_sellin
        UNION ALL
        SELECT * FROM tb_zeroes
        ORDER BY 1, 2, 3
    """)
    n_real  = con.sql("SELECT COUNT(*) FROM tb_sellin").fetchone()[0]
    n_total = con.sql("SELECT COUNT(*) FROM tb_sellin_zeroes").fetchone()[0]
    print(f"filas reales : {n_real:,}   filas totales: {n_total:,}   "
         f"({100*(n_total-n_real)/n_total:.0f}% ceros)")

    dup = con.sql("""
        SELECT COUNT(*) FROM (
            SELECT customer_id, product_id, periodo, COUNT(*) AS n
            FROM tb_sellin_zeroes GROUP BY 1, 2, 3 HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    assert dup == 0, f"hay {dup} combinaciones (cliente, producto, periodo) duplicadas"
    print("chequeo: sin duplicados (cliente, producto, periodo) -> ok")

    con.execute(f"""
        COPY (SELECT * FROM tb_sellin_zeroes ORDER BY 1, 2, 3)
        TO '{path_pre}' (FORMAT parquet)
    """)
    con.close()
    print(f"Guardado: {path_pre}   [{time.time()-t0:.0f}s]")


## 2) Feature engineering — portado de `02_FE.ipynb` + racha NUEVA

Shares jerarquicos (cat1/cat2/cat3/mercado), lags, medias moviles
(3/6/**12**, se agrego 12 para `delta_mean_12`), deltas de share, indices,
**racha consecutiva** (NUEVO: meses SEGUIDOS con venta, no frecuencia),
recencia, peso acumulado, vecinos por correlacion, target `clase_tn`.
Tambien cacheado — si el parquet ya existe, se lee directo.


In [ ]:
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE_SIN_ESCALAR = (f"features_sin_escalar_grpClienteProducto{_tgt}_{L}lags_share_{H}h"
                      f"_vec{PARAM['n_vecinos']}_otroPipe.parquet")
path_fe = DIR_FE / NOMBRE_SIN_ESCALAR

if path_fe.exists():
    print(f"cache encontrada: {path_fe.name} -> se salta el FE")
else:
    t0 = time.time()
    con = duckdb.connect()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_raw AS
        SELECT * FROM read_parquet('{path_pre}')
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE prod AS
        SELECT * FROM read_csv('{DIR_RAW / "tb_productos.txt"}', delim='\t', header=true)
    """)
    n_raw = con.sql("SELECT COUNT(*) FROM panel_raw").fetchone()[0]
    print(f"preprocesado: {n_raw:,} filas   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_0 AS
        SELECT {KEYS_SQL}, periodo,
               SUM(tn) AS tn,
               SUM(cust_request_tn) AS req_tn,
               SUM(cust_request_qty) AS req_qty,
               MAX(plan_precios_cuidados) AS precios_cuidados,
               ((periodo // 100) * 12 + (periodo % 100)) AS m
        FROM panel_raw GROUP BY {KEYS_SQL}, periodo
    """)
    n0 = con.sql("SELECT COUNT(*) FROM panel_0").fetchone()[0]
    rango = con.sql("SELECT MIN(periodo), MAX(periodo) FROM panel_0").fetchone()
    print(f"panel agregado: {n0:,} filas · rango {rango[0]} -> {rango[1]}   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT p.*,
               MIN(m) OVER (PARTITION BY {KEYS_SQL}) AS m_nace,
               MAX(m) OVER (PARTITION BY {KEYS_SQL}) AS m_muere,
               pr.cat1, pr.cat2, pr.cat3, pr.brand, pr.sku_size
        FROM panel_0 p LEFT JOIN prod pr USING (product_id)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT *, CASE WHEN m >= m_nace THEN m - m_nace ELSE -1 END AS edad_cliente_producto
        FROM panel_1
    """)
    n_ceros = con.sql("SELECT COUNT(*) FROM panel_1 WHERE tn = 0").fetchone()[0]
    print(f"ceros de tn: {n_ceros:,} ({100*n_ceros/n0:.0f}%)   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tot_prod AS
        SELECT product_id, m, SUM(tn) AS tn_prod, COUNT(*) AS n_clientes_prod
        FROM panel_1 GROUP BY product_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli AS
        SELECT customer_id, m, SUM(tn) AS tn_cli, COUNT(*) AS n_productos_cli
        FROM panel_1 GROUP BY customer_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE vida_prod AS
        SELECT product_id, MIN(m) AS m_nace_prod FROM tot_prod GROUP BY product_id
    """)
    con.execute("""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT p.*,
               CASE WHEN p.m >= v.m_nace_prod THEN p.m - v.m_nace_prod ELSE -1 END AS edad_producto
        FROM panel_1 p LEFT JOIN vida_prod v USING (product_id)
    """)

    for niv in PARAM['niveles_share']:
        if niv == 'mercado':
            con.execute("""
                CREATE OR REPLACE TABLE tot_mercado AS
                SELECT m, SUM(tn_prod) AS tn_mercado FROM tot_prod GROUP BY m
            """)
        else:
            con.execute(f"""
                CREATE OR REPLACE TABLE tot_{niv} AS
                SELECT pr.{niv} AS {niv}, tp.m, SUM(tp.tn_prod) AS tn_{niv}, COUNT(*) AS n_prod_{niv}
                FROM tot_prod tp LEFT JOIN prod pr USING (product_id)
                GROUP BY pr.{niv}, tp.m
            """)
    print(f"tot_prod: {con.sql('SELECT COUNT(*) FROM tot_prod').fetchone()[0]:,} filas   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT p.*, tp.tn_prod, tp.n_clientes_prod, tc.tn_cli, tc.n_productos_cli,
               CASE WHEN ABS(tc.tn_cli) > 1e-9 THEN p.tn / tc.tn_cli ELSE 0.0 END AS sh_prod_en_cli,
               CASE WHEN ABS(tp.tn_prod) > 1e-9 THEN p.tn / tp.tn_prod ELSE 0.0 END AS sh_cli_en_prod
        FROM panel_1 p
        LEFT JOIN tot_prod tp USING (product_id, m)
        LEFT JOIN tot_cli tc USING (customer_id, m)
    """)
    SHARES = ["sh_prod_en_cli", "sh_cli_en_prod"]

    for niv in PARAM['niveles_share']:
        if niv == 'mercado':
            con.execute("""
                CREATE OR REPLACE TABLE df AS
                SELECT d.*, tm.tn_mercado,
                       CASE WHEN ABS(tm.tn_mercado) > 1e-9 THEN d.tn_prod / tm.tn_mercado ELSE 0.0 END AS sh_prod_en_mercado
                FROM df d LEFT JOIN tot_mercado tm USING (m)
            """)
            SHARES.append("sh_prod_en_mercado")
        else:
            con.execute(f"""
                CREATE OR REPLACE TABLE df AS
                SELECT d.*, t.tn_{niv},
                       CASE WHEN ABS(t.tn_{niv}) > 1e-9 THEN d.tn_prod / t.tn_{niv} ELSE 0.0 END AS sh_prod_en_{niv}
                FROM df d LEFT JOIN tot_{niv} t ON d.{niv} = t.{niv} AND d.m = t.m
            """)
            SHARES.append(f"sh_prod_en_{niv}")
    print(f"{len(SHARES)} shares: {SHARES}   [{time.time()-t0:.0f}s]")

    mn, mx = con.sql("""
        SELECT MIN(s), MAX(s) FROM (
            SELECT customer_id, m, SUM(sh_prod_en_cli) AS s FROM df GROUP BY 1, 2
        )
    """).fetchone()
    assert abs(mx - 1.0) < 1e-6, "sh_prod_en_cli no suma 1 en algun cliente-mes"
    if 'cat3' in PARAM['niveles_share']:
        mn3, mx3 = con.sql("""
            SELECT MIN(s), MAX(s) FROM (
                SELECT cat3, m, SUM(sh_prod_en_cat3) AS s
                FROM (SELECT DISTINCT product_id, cat3, m, sh_prod_en_cat3 FROM df)
                GROUP BY 1, 2
            )
        """).fetchone()
        assert abs(mx3 - 1.0) < 1e-6, "sh_prod_en_cat3 no suma 1 en algun cat3-mes"
    print("chequeo de shares OK")

    t0 = time.time()
    lag_exprs = [f"LAG(tn, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS tn_lag{k}"
                for k in range(1, L + 1)]
    ma_exprs = [f"AVG(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
               f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS tn_ma{w}"
               for w in PARAM['ventanas_ma']]
    qty_ma_exprs = [f"AVG(req_qty) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                   f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS qty_ma{w}"
                   for w in PARAM['ventanas_ma'][:2]]
    otros = [
        f"LAG(req_qty, 1) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS qty_lag1",
        f"AVG(req_tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
        f"ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS reqtn_ma3",
        f"MAX(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
        f"ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_pico_hasta_aca",
    ]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(lag_exprs + ma_exprs + qty_ma_exprs + otros)} FROM df")
    print(f"lags 1..{L} + medias moviles {PARAM['ventanas_ma']} agregados   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    delta_exprs = []
    for s in SHARES:
        delta_exprs += [f"LAG({s}, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS {s}_lag{k}"
                        for k in range(1, PARAM['lags_share'] + 1)]
        delta_exprs += [f"AVG({s}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                        f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS {s}_ma{w}"
                        for w in (3, 6)]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(delta_exprs)} FROM df")

    d_exprs = []
    for s in SHARES:
        d_exprs.append(f"({s} - {s}_lag1) AS {s}_d1")
        d_exprs.append(f"({s} - {s}_ma3) AS {s}_dma3")
        if PARAM['lags_share'] >= 3:
            d_exprs.append(f"({s} - {s}_lag3) AS {s}_d3")
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(d_exprs)} FROM df")

    TECHO = PARAM['techo_indice']
    idx_exprs = [
        f"CASE WHEN ABS(tn_lag1) > 1e-9 THEN LEAST(GREATEST(tn / tn_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_tn_mom",
        f"CASE WHEN ABS(tn_ma3) > 1e-9 THEN LEAST(GREATEST(tn / tn_ma3, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_ma3",
        f"CASE WHEN ABS(tn_pico_hasta_aca) > 1e-9 THEN LEAST(GREATEST(tn / tn_pico_hasta_aca, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_pico",
        f"CASE WHEN ABS(qty_lag1) > 1e-9 THEN LEAST(GREATEST(req_qty / qty_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_qty_mom",
    ]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(idx_exprs)} FROM df")
    print(f"deltas de share + 4 indices agregados. [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("CREATE OR REPLACE TABLE df AS SELECT *, CAST(tn > 0 AS TINYINT) AS vendio FROM df")
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               AVG(CAST(vendio AS DOUBLE)) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS frac_meses_con_venta_6,
               SUM(vendio) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS meses_con_venta_3,
               (periodo % 100) AS mes_del_anio,
               CAST(edad_cliente_producto BETWEEN 0 AND 6 AS TINYINT) AS es_nuevo
        FROM df
    """)

    # ── Racha consecutiva (NUEVO): meses SEGUIDOS con venta contando hacia atras
    # desde el mes actual, capeada en cada ventana. Distinta de frac_meses_con_venta_6
    # (frecuencia en la ventana, no le importa si hay huecos) -- aca un solo mes sin
    # venta corta la racha a cero. COALESCE(LAG(...), 0): sin historia previa dentro
    # del par (borde de la ventana) se interpreta como 'no vendio' (no hay racha).
    def _racha_expr(w):
        terminos = []
        for k in range(1, w + 1):
            factores = ["vendio"] + [
                f"COALESCE(LAG(vendio, {j}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m), 0)"
                for j in range(1, k)
            ]
            terminos.append("(" + " * ".join(factores) + ")")
        return " + ".join(terminos)

    racha_exprs = [f"({_racha_expr(w)}) AS racha_{w}" for w in PARAM['ventanas_racha']]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(racha_exprs)} FROM df")
    print(f"racha consecutiva agregada: {[f'racha_{w}' for w in PARAM['ventanas_racha']]}   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               LAST_VALUE(CASE WHEN tn > 0 THEN m END IGNORE NULLS) OVER (
                   PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
               ) AS _ultimo_m_con_venta_prev
        FROM df
    """)
    con.execute("CREATE OR REPLACE TABLE df AS SELECT * EXCLUDE (_ultimo_m_con_venta_prev), "
               "(m - _ultimo_m_con_venta_prev) AS meses_sin_compra FROM df")
    print(f"meses_sin_compra agregado.   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tn_total_mes AS
        SELECT m, SUM(tn) AS tn_total_mes,
               SUM(SUM(tn)) OVER (ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_total_acum
        FROM panel_1 GROUP BY m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_prod_acum AS
        SELECT tp.product_id, tp.m,
               CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                    THEN SUM(tp.tn_prod) OVER (PARTITION BY tp.product_id ORDER BY tp.m
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                    ELSE 0.0 END AS peso_producto_acum
        FROM tot_prod tp LEFT JOIN tn_total_mes tm USING (m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, tpa.peso_producto_acum
        FROM df d LEFT JOIN tot_prod_acum tpa USING (product_id, m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli_acum AS
        SELECT tc.customer_id, tc.m,
               CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                    THEN SUM(tc.tn_cli) OVER (PARTITION BY tc.customer_id ORDER BY tc.m
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                    ELSE 0.0 END AS peso_cliente_acum
        FROM tot_cli tc LEFT JOIN tn_total_mes tm USING (m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, tca.peso_cliente_acum
        FROM df d LEFT JOIN tot_cli_acum tca USING (customer_id, m)
    """)
    print(f"peso_producto_acum y peso_cliente_acum agregados.   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    M_CORTE = (PARAM['mes_corte_vecinos'] // 100) * 12 + (PARAM['mes_corte_vecinos'] % 100)
    N_VEC = PARAM['n_vecinos']

    wide = (con.sql(f"""
                SELECT product_id, m, tn_prod FROM tot_prod WHERE m < {M_CORTE}
            """).pl()
               .pivot(on="product_id", index="m", values="tn_prod")
               .sort("m")
               .drop("m"))
    corr = wide.to_pandas().corr(method="spearman")

    vecinos_rows = []
    for p in corr.columns:
        s = corr[p].drop(labels=[p]).dropna()
        if s.empty:
            continue
        for rank, (vecino, r) in enumerate(s.sort_values().head(N_VEC).items(), start=1):
            vecinos_rows.append({"product_id": p, "tipo": "sustituto", "rank": rank,
                                 "vecino_id": vecino, "corr": float(r)})
        for rank, (vecino, r) in enumerate(s.sort_values(ascending=False).head(N_VEC).items(), start=1):
            vecinos_rows.append({"product_id": p, "tipo": "complementario", "rank": rank,
                                 "vecino_id": vecino, "corr": float(r)})

    vecinos_df = pd.DataFrame(vecinos_rows)
    con.register("vecinos_pl", vecinos_df)
    con.execute("""
        CREATE OR REPLACE TABLE vecinos AS
        SELECT CAST(product_id AS BIGINT) AS product_id, tipo, rank,
               CAST(vecino_id AS BIGINT) AS vecino_id, corr
        FROM vecinos_pl
    """)
    con.unregister("vecinos_pl")
    print(f"vecinos calculados para {con.sql('SELECT COUNT(DISTINCT product_id) FROM vecinos').fetchone()[0]} "
         f"productos (corte {PARAM['mes_corte_vecinos']})   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE feat_vecinos AS
        SELECT v.product_id, v.tipo, tp.m, AVG(tp.tn_prod) AS tn_vecino_prom
        FROM vecinos v LEFT JOIN tot_prod tp ON tp.product_id = v.vecino_id
        GROUP BY v.product_id, v.tipo, tp.m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE feat_vecinos_piv AS
        SELECT product_id, m,
               MAX(CASE WHEN tipo = 'sustituto' THEN tn_vecino_prom END) AS tn_sustitutos_prom,
               MAX(CASE WHEN tipo = 'complementario' THEN tn_vecino_prom END) AS tn_complementarios_prom
        FROM feat_vecinos GROUP BY product_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*,
               COALESCE(fv.tn_sustitutos_prom, 0.0) AS tn_sustitutos_prom,
               COALESCE(fv.tn_complementarios_prom, 0.0) AS tn_complementarios_prom
        FROM df d LEFT JOIN feat_vecinos_piv fv USING (product_id, m)
    """)
    print(f"tn_sustitutos_prom / tn_complementarios_prom agregados.   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               LEAD(tn, {H}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS clase_tn,
               (((m + {H} - 1) // 12) * 100) + ((m + {H} - 1) % 12) + 1 AS periodo_objetivo
        FROM df
    """)
    _sup = con.sql("SELECT COUNT(*) FROM df WHERE clase_tn IS NOT NULL").fetchone()[0]
    _tot = con.sql("SELECT COUNT(*) FROM df").fetchone()[0]
    print(f"filas con target: {_sup:,}   filas de inferencia: {_tot - _sup:,}   [{time.time()-t0:.0f}s]")

    errores = []
    def chk(ok, msg):
        print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
        if not ok:
            errores.append(msg)

    print("CONTROL DE DATA LEAKAGE (FE)")
    print("=" * 74)
    _una = con.sql(f"""
        SELECT {KEYS_SQL} FROM df WHERE clase_tn IS NOT NULL
        GROUP BY {KEYS_SQL} ORDER BY COUNT(*) DESC LIMIT 1
    """).fetchone()
    _k = dict(zip(KEYS, _una))
    _where = " AND ".join(f"{c} = {v}" for c, v in _k.items())
    _serie = con.sql(f"SELECT m, tn, clase_tn FROM df WHERE {_where} ORDER BY m").pl()
    _tn, _cl = _serie["tn"].to_list(), _serie["clase_tn"].to_list()
    _malos = [i for i in range(len(_tn) - H)
             if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
    chk(not _malos, f"clase_tn[i] == tn[i+{H}] en la serie {_k}")

    info = con.sql("DESCRIBE df").fetchall()
    tipos = {r[0]: r[1].upper().split("(")[0] for r in info}
    PROHIBIDAS = {"clase_tn", "periodo_objetivo", "m_muere", "m", "periodo", "m_nace"} | set(KEYS)
    FEATURES_FE = [c for c in tipos if c not in PROHIBIDAS]
    chk(not (set(FEATURES_FE) & {"clase_tn", "periodo_objetivo"}), "el target no esta entre las features")
    chk("m_muere" not in FEATURES_FE, "m_muere no es feature")
    chk("m_nace" not in FEATURES_FE, "m_nace no es feature")

    NUMERIC = {"TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "UTINYINT",
              "USMALLINT", "UINTEGER", "UBIGINT", "FLOAT", "DOUBLE", "DECIMAL"}
    NUM_FEATURES = [c for c in FEATURES_FE if tipos[c] in NUMERIC]
    _sel = ", ".join(f'CORR(clase_tn, "{c}") AS "{c}"' for c in NUM_FEATURES)
    _res = con.sql(f"SELECT {_sel} FROM df WHERE clase_tn IS NOT NULL").fetchone()
    _sosp = [(c, round(v, 5)) for c, v in zip(NUM_FEATURES, _res) if v is not None and abs(v) > 0.999]
    chk(not _sosp, f"ninguna de las {len(NUM_FEATURES)} features numericas correlaciona >0.999 con clase_tn  {_sosp}")

    _s2 = con.sql(f"SELECT m, tn, tn_ma3 FROM df WHERE {_where} ORDER BY m").pl()
    _tn2, _ma = _s2["tn"].to_list(), _s2["tn_ma3"].to_list()
    _err_ma = max((abs(_ma[i] - sum(_tn2[i-2:i+1]) / 3)
                  for i in range(2, len(_tn2)) if _ma[i] is not None), default=0.0)
    chk(_err_ma < 1e-9, f"tn_ma3[t] == promedio(tn[t-2..t]): error maximo {_err_ma:.2e}")

    # chequeo de la racha: en la misma serie, racha_3 tiene que ser <=3 y solo > 0
    # si vendio en el mes actual, y coincidir con el conteo manual
    _s3 = con.sql(f"SELECT m, vendio, racha_3 FROM df WHERE {_where} ORDER BY m").pl()
    _v3, _r3 = _s3["vendio"].to_list(), _s3["racha_3"].to_list()
    _r3_manual = []
    _run = 0
    for v in _v3:
        _run = (_run + 1) if v else 0
        _r3_manual.append(min(_run, 3))
    _err_racha = max((abs(a - b) for a, b in zip(_r3, _r3_manual)), default=0)
    chk(_err_racha == 0, f"racha_3 coincide con el conteo manual de racha consecutiva (error max {_err_racha})")

    chk(PARAM['mes_corte_vecinos'] <= max(PARAM['meses_train']),
       f"mes_corte_vecinos no supera el fin de meses_train")

    print("=" * 74)
    if errores:
        raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
    print(f"Control superado. {len(FEATURES_FE)} features ({len(NUM_FEATURES)} numericas).")

    t0 = time.time()
    DROP = {"m_nace", "m_muere"}
    CTX_F64 = {"clase_tn", "tn0"}
    exprs = []
    for name, tipo, *_r in info:
        if name in DROP:
            continue
        tipo_base = tipo.upper().split("(")[0]
        out_name = "tn0" if name == "tn" else name
        if tipo_base == "DOUBLE" and out_name not in CTX_F64:
            exprs.append(f'CAST("{name}" AS FLOAT) AS "{out_name}"')
        else:
            exprs.append(f'"{name}" AS "{out_name}"')
    select_sql = ",\n       ".join(exprs)

    con.execute(f"""
        COPY (SELECT {select_sql} FROM df ORDER BY {KEYS_SQL}, m)
        TO '{path_fe}' (FORMAT parquet)
    """)
    con.close()
    print(f"Guardado: {path_fe}   {_tot:,} filas x {len(exprs)} columnas   [{time.time()-t0:.0f}s]")


## 3) Carga + split train/val/test/infer + control de leakage

Se carga el parquet de FE (Float64 -> Float32 salvo columnas de contexto),
se separan filas supervisadas (`clase_tn` no nulo) de las de inferencia, y
se valida el split ANTES de tocar nada de escalado/target -- estos son fijos
para toda la grilla, no varian por combo.


In [ ]:
t0 = time.time()
CTX_F64 = {'clase_tn', 'tn0'}

lf = pl.scan_parquet(path_fe)
_schema = lf.collect_schema()
COLS_ALL = list(_schema.keys())
_f64 = [c for c, t in _schema.items() if t == pl.Float64]
_a_f32 = [c for c in _f64 if c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
print(f"Dataset: {len(COLS_ALL)} columnas   Periodos: {periodos[0]} -> {periodos[-1]} ({len(periodos)} meses)")

MESES_INFER = periodos[-H:]
df_infer = lf.filter(pl.col('clase_tn').is_null() & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup = lf.filter(pl.col('clase_tn').is_not_null()).collect()
print(f"Supervisadas: {df_sup.height:,}   Inferencia: {df_infer.height:,} -> "
     f"periodos {sorted(df_infer['periodo'].unique().to_list())}")
print(f"[{time.time()-t0:.0f}s]")

periodos_sup = sorted(df_sup['periodo'].unique().to_list())
set_sup = set(periodos_sup)
MESES_TRAIN = sorted(set(PARAM['meses_train']) & set_sup)
MESES_VAL   = sorted(set(PARAM['meses_val'])   & set_sup)
MESES_TEST  = sorted(set(PARAM['meses_test'])  & set_sup)
for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(f"{nombre} quedo vacio. Disponibles: {periodos_sup[0]}..{periodos_sup[-1]}")
    print(f"{nombre:6s} ({len(ms):2d} meses): {ms[0]} .. {ms[-1]}")

leak = {'errores': [], 'ok': []}
def _err(msg):
    leak['errores'].append(msg); print(f"  [ERROR] {msg}")
def _ok(msg):
    leak['ok'].append(msg); print(f"  [ok]    {msg}")

print("CONTROL DE DATA LEAKAGE (split)")
print("=" * 72)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, 'train', 'val'), (MESES_VAL, MESES_TEST, 'val', 'test')):
    gap = a_indice_mes(min(b)) - a_indice_mes(max(a))
    (_ok if gap >= H else _err)(f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es)"
                                + ("" if gap >= H else f" < horizonte {H}"))
if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_indice_mes(min(MESES_TEST)) - a_indice_mes(max(MESES_TRAIN + MESES_VAL))
    (_ok if gap_tv >= H else _err)(f"gap (train+val) -> test = {gap_tv}" + ("" if gap_tv >= H else f" < {H}"))
for (na, a), (nb, b) in ((('train', MESES_TRAIN), ('val', MESES_VAL)),
                        (('train', MESES_TRAIN), ('test', MESES_TEST)),
                        (('val', MESES_VAL),     ('test', MESES_TEST))):
    inter = sorted(set(a) & set(b))
    (_err if inter else _ok)(f"{na} y {nb}" + (f" comparten {inter}" if inter else " son disjuntos"))
if max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST):
    _ok("orden cronologico correcto: train < val < test")
else:
    _err("orden cronologico incorrecto")
periodos_infer = set(df_infer['periodo'].unique().to_list())
solapa = sorted(periodos_infer & (set(MESES_TRAIN) | set(MESES_VAL) | set(MESES_TEST)))
(_err if solapa else _ok)("periodos de inferencia" + (f" solapan: {solapa}" if solapa else " no se usan para entrenar/validar/testear"))
print("=" * 72)
if leak['errores']:
    raise RuntimeError(f"Control de leakage FALLIDO: {leak['errores']}")
print(f"Control superado. {df_sup.height:,} filas supervisadas, {df_infer.height:,} de inferencia.")


## 4) Filtro del universo de entrenamiento (80% de demanda + magicos) + fallback

El share por categoria y los vecinos YA se calcularon sobre el panel
COMPLETO (arriba) -- este filtro solo acota que pares `(customer_id,
product_id)` entran al Optuna/entrenamiento sofisticado de la grilla. Los
pares que quedan afuera igual tienen que aparecer en el submit: se les
aplica un fallback barato (`tn_ma3`, ya calculado en el FE) sobre el panel
completo.


In [ ]:
t0 = time.time()

demanda_pares = (df_sup.group_by(['customer_id', 'product_id'])
                       .agg(pl.col('tn0').sum().alias('tn_total'))
                       .sort('tn_total', descending=True)
                       .with_columns(
                           (pl.col('tn_total').cum_sum() - pl.col('tn_total')).alias('_acum_antes')))
_total_demanda = demanda_pares['tn_total'].sum()
demanda_pares = demanda_pares.with_columns(
    (pl.col('_acum_antes') / _total_demanda if _total_demanda > 0 else pl.lit(0.0)).alias('_frac_acum_antes'))
pares_volumen = demanda_pares.filter(pl.col('_frac_acum_antes') < PARAM['pct_demanda_cubrir'])

print(f"pares totales: {demanda_pares.height:,}   "
     f"pares que cubren el {100*PARAM['pct_demanda_cubrir']:.0f}% de la demanda: {pares_volumen.height:,} "
     f"({100*pares_volumen.height/demanda_pares.height:.0f}%)")
print(f"demanda cubierta por esos pares: "
     f"{100*pares_volumen['tn_total'].sum()/_total_demanda:.1f}%   [{time.time()-t0:.0f}s]")

PRODUCTOS_MAGICOS = None
if PARAM['archivo_productos_magicos']:
    path_mag = DIR_FE / PARAM['archivo_productos_magicos']
    if not path_mag.exists():
        print(f"no encontre {path_mag} -- corre nat_exp/residuos_2.ipynb primero. "
             f"Se sigue solo con el filtro de volumen.")
    else:
        _meta_mag = _leer_json_reintentando(path_mag)
        PRODUCTOS_MAGICOS = set(_meta_mag['product_ids'])
        print(f"productos magicos leidos de {path_mag.name}: {len(PRODUCTOS_MAGICOS)}")

pares_universo = pares_volumen
if PRODUCTOS_MAGICOS is not None:
    _con_magicos = pares_volumen.filter(pl.col('product_id').is_in(PRODUCTOS_MAGICOS))
    if _con_magicos.height == 0:
        print(f"AVISO: la interseccion con productos magicos ({len(PRODUCTOS_MAGICOS)} ids) dio "
             f"0 pares dentro del top {100*PARAM['pct_demanda_cubrir']:.0f}% de demanda -- "
             f"se sigue solo con el filtro de volumen (sin magicos).")
    else:
        pares_universo = _con_magicos
        print(f"pares tras interseccion con magicos: {pares_universo.height:,} "
             f"(de {pares_volumen.height:,} solo-volumen)")

if pares_universo.height == 0:
    raise RuntimeError("El universo de entrenamiento quedo vacio incluso solo con volumen. "
                       "Revisa PARAM['pct_demanda_cubrir'].")

UNIVERSO = pares_universo.select(['customer_id', 'product_id']).unique()
df_sup_universo = df_sup.join(UNIVERSO, on=['customer_id', 'product_id'], how='inner')
df_infer_universo = df_infer.join(UNIVERSO, on=['customer_id', 'product_id'], how='inner')
print(f"\ndf_sup_universo: {df_sup_universo.height:,} filas (de {df_sup.height:,})")
print(f"df_infer_universo: {df_infer_universo.height:,} filas (de {df_infer.height:,})")

# ── Fallback para los pares EXCLUIDOS del universo (tn_ma3, sobre el panel
# completo -- la entrega tiene que cubrir todos los productos oficiales) ──
FALLBACK_INFER = (df_infer.join(UNIVERSO, on=['customer_id', 'product_id'], how='anti')
                          .select('product_id', 'customer_id', 'periodo', 'periodo_objetivo',
                                  pl.col('tn_ma3').fill_null(0.0).alias('tn_pred_fallback')))
print(f"fallback (fuera del universo, inferencia): {FALLBACK_INFER.height:,} filas   [{time.time()-t0:.0f}s]")


## 5) Los 5 targets

Todos usan el mismo esquema afin `norm = (valor-B0)/B1` (invertible exacto,
`valor = norm*B1 + B0`) que el resto de la sesion -- incluidos `rango`
(min-max) y `normalpower` (IQR robusto, nombre pedido por la usuaria; NO es
un power-transform real porque eso rompe la reconstruccion afin). Los 3
targets escalado-nativos (`ton_norm`, `delta_ton_norm`, `log_ton_norm`)
varian segun el metodo de escalado; los 2 fijos (`delta_mean_12`,
`delta_reg`) no -- ver el plan sobre por que la grilla es de 20 combos y no
30 (LightGBM es invariante a reescalados afines de los features de
entrada).


In [ ]:
LAG_RAW = ["tn0"] + [f"tn_lag{k}" for k in range(1, L + 1)]

# 'rango' (max-min) y 'normalpower' (IQR) pueden dar un B1 MINUSCULO pero no
# exactamente cero en datos reales dispersos (ventanas casi todas en cero con
# una sola venta chica) -- eso no lo agarra un chequeo de "B1 == 0", y dividir
# por un numero asi de chico dispara el target normalizado (y despues la
# reconstruccion) a valores absurdos. 'mediana'/'zscore' no sufren esto porque
# la mediana/el desvio de una ventana mayormente-cero cae directo en 0 (si cae
# justo en un valor chico pero no cero, este mismo piso los protege igual).
# 1e-3 tn (1 kg) como piso: cualquier spread mas chico que eso no es señal real.
B1_EPSILON = 1e-3


def _calcular_b0_b1(metodo, lag_cols, ma_col):
    if metodo == "mediana":
        return pl.lit(0.0), pl.concat_list(lag_cols).list.median()
    if metodo == "media":
        return pl.lit(0.0), pl.mean_horizontal(lag_cols)
    if metodo == "zscore":
        return pl.mean_horizontal(lag_cols), pl.concat_list(lag_cols).list.std()
    if metodo == "rolling_mean":
        return pl.lit(0.0), pl.col(ma_col)
    if metodo == "rango":
        _l = pl.concat_list(lag_cols)
        return _l.list.min(), _l.list.max() - _l.list.min()
    if metodo == "normalpower":
        # IQR robusto (nombre pedido por la usuaria; NO es Yeo-Johnson/Box-Cox real --
        # eso no es afin y rompe la reconstruccion exacta a toneladas).
        _l = pl.concat_list(lag_cols)
        p25 = _l.list.eval(pl.element().quantile(0.25)).list.first()
        p75 = _l.list.eval(pl.element().quantile(0.75)).list.first()
        return _l.list.median(), p75 - p25
    raise ValueError(f"metodo de escalado no soportado: {metodo}")


def preparar_target_escalado(df_in, target_name, metodo):
    """ton_norm | delta_ton_norm | log_ton_norm. Devuelve (df_out, kind) con
    columnas B0, B1, y (target del combo) agregadas. kind se usa despues para
    saber como reconstruir_target() vuelve a toneladas."""
    usa_log = target_name == 'log_ton_norm'
    out = df_in
    if usa_log:
        out = out.with_columns([pl.col(c).clip(lower_bound=0.0).log1p().alias(f"__log_{c}")
                                for c in LAG_RAW])
        out = out.with_columns(pl.col(f"tn_ma{PARAM['ventana_escalado_rolling']}")
                               .clip(lower_bound=0.0).log1p().alias('__log_tn_ma'))
        lag_cols, ma_col = [f"__log_{c}" for c in LAG_RAW], '__log_tn_ma'
        clase_src, tn0_src = pl.col('clase_tn').clip(lower_bound=0.0).log1p(), pl.col('__log_tn0')
    else:
        lag_cols, ma_col = LAG_RAW, f"tn_ma{PARAM['ventana_escalado_rolling']}"
        clase_src, tn0_src = pl.col('clase_tn'), pl.col('tn0')

    B0, B1 = _calcular_b0_b1(metodo, lag_cols, ma_col)
    out = out.with_columns(B0.alias('B0'), B1.alias('B1'))
    b1_safe = (pl.when((pl.col('B1').abs() < B1_EPSILON) | pl.col('B1').is_null())
                 .then(1.0).otherwise(pl.col('B1')))
    out = out.with_columns(
        ((tn0_src - pl.col('B0')) / b1_safe).alias('_tn0_norm'),
        ((clase_src - pl.col('B0')) / b1_safe).alias('_clase_norm'))

    if target_name == 'delta_ton_norm':
        out = out.with_columns((pl.col('_clase_norm') - pl.col('_tn0_norm')).alias('y'))
        kind = 'delta_log' if usa_log else 'delta_nivel'
    else:
        out = out.with_columns(pl.col('_clase_norm').alias('y'))
        kind = 'log' if usa_log else 'nivel'
    return out, kind


def preparar_target_delta_mean12(df_in):
    out = df_in.with_columns(
        pl.col('tn_ma12').fill_null(0.0).alias('B0'),
        pl.lit(1.0).alias('B1'),
    ).with_columns((pl.col('clase_tn') - pl.col('B0')).alias('y'))
    return out, 'residuo_fijo'


COLS_LIN_REG = ["tn0"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def fit_ridge_delta_reg(df_train_pl):
    from sklearn.linear_model import Ridge
    X = df_train_pl.select(COLS_LIN_REG).fill_null(0.0).to_numpy()
    y = df_train_pl['clase_tn'].to_numpy()
    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X, y)
    return r


def preparar_target_delta_reg(df_in, ridge_model):
    X = df_in.select(COLS_LIN_REG).fill_null(0.0).to_numpy()
    b0 = np.maximum(ridge_model.predict(X), 0.0)
    out = df_in.with_columns(pl.Series('B0', b0), pl.lit(1.0).alias('B1'))
    out = out.with_columns((pl.col('clase_tn') - pl.col('B0')).alias('y'))
    return out, 'residuo_fijo'


def reconstruir_target(pred, df_ctx, kind) -> np.ndarray:
    """Inverso exacto de preparar_target_*: vuelve a toneladas."""
    pred = np.asarray(pred, dtype=np.float64)
    if kind == 'residuo_fijo':
        return np.maximum(pred + df_ctx['B0'].to_numpy().astype(np.float64), 0.0)
    if kind.startswith('delta_'):
        pred = pred + df_ctx['_tn0_norm'].to_numpy().astype(np.float64)
        kind = kind[len('delta_'):]
    B0 = df_ctx['B0'].to_numpy().astype(np.float64)
    B1 = df_ctx['B1'].to_numpy().astype(np.float64)
    B1_safe = np.where((np.abs(B1) < B1_EPSILON) | ~np.isfinite(B1), 1.0, B1)
    val = pred * B1_safe + B0
    if kind == 'log':
        # expm1 amplifica exponencialmente: un trial de Optuna sin entrenar bien
        # (early stopping recien arrancando, hiperparametros malos) puede predecir
        # un valor grande en espacio log-normalizado y desbordar a inf -- clip
        # antes de exponenciar (20 en log1p-space ya son ~5e8 toneladas, muy por
        # encima de cualquier prediccion real; solo protege contra el desborde).
        val = np.expm1(np.clip(val, -20.0, 20.0))
    return np.maximum(val, 0.0)


print("funciones de target listas: preparar_target_escalado / _delta_mean12 / _delta_reg / reconstruir_target")


### Chequeo round-trip de los 20 combos (antes de gastar tiempo en Optuna)


In [ ]:
t0 = time.time()
_muestra = df_sup_universo.filter(pl.col('periodo').is_in(MESES_TRAIN)).head(2000)
_ridge_chk = fit_ridge_delta_reg(_muestra)

print(f"{'target':16s} {'escalado':14s} {'error max reconstruccion':>26s}")
print("-" * 60)
for target_name, metodo in GRILLA:
    if target_name in PARAM['targets_escalado_nativos']:
        out, kind = preparar_target_escalado(_muestra, target_name, metodo)
    elif target_name == 'delta_mean_12':
        out, kind = preparar_target_delta_mean12(_muestra)
    elif target_name == 'delta_reg':
        out, kind = preparar_target_delta_reg(_muestra, _ridge_chk)
    else:
        raise ValueError(target_name)
    rec = reconstruir_target(out['y'].to_numpy(), out, kind)
    err = float(np.max(np.abs(rec - out['clase_tn'].clip(lower_bound=0.0).to_numpy())))
    print(f"{target_name:16s} {str(metodo):14s} {err:26.2e}")
    assert err < 1e-6, f"round-trip roto para target={target_name} escalado={metodo} (kind={kind}): error {err}"
print(f"\n20/20 combos reconstruyen a toneladas con error <1e-6.   [{time.time()-t0:.0f}s]")


## 6) Motor de Optuna + WAPE (portado de `pipe_nuevo/06_Optuna.ipynb`)

Mismo motor ya probado esta sesion: espacio de hiperparametros con techo de
arboles + early stopping (no se sortea `n_estimators`, se reusa
`best_iteration_` del mejor trial para los reentrenos), WAPE agregado por
producto. Generalizado para trabajar con la columna `y` (el target del
combo activo) y `reconstruir_target(..., kind)` en vez de una unica
`reconstruir_nivel`.


In [ ]:
COLS_ID = [c for c in ['product_id', 'customer_id', 'periodo', 'm', 'periodo_objetivo']
          if c in df_sup_universo.columns]
PROHIBIDAS_BASE = set(COLS_ID) | {'clase_tn'}
FEATURES = [c for c in df_sup_universo.columns if c not in PROHIBIDAS_BASE]
CAT_PEDIDAS = ['cat1', 'cat2', 'cat3', 'brand']
TIPOS_TEXTO = (pl.Utf8, pl.String, pl.Categorical, pl.Enum, pl.Boolean)
CAT_AUTO = [c for c in FEATURES if df_sup_universo.schema[c] in TIPOS_TEXTO and c not in CAT_PEDIDAS]
CAT_FEATURES = [c for c in CAT_PEDIDAS if c in FEATURES] + CAT_AUTO
print(f"FEATURES: {len(FEATURES)}   CAT_FEATURES: {CAT_FEATURES}")


def wape(y_real_tn, y_pred_tn, product_ids=None, por_producto=True) -> float:
    y_real = np.asarray(y_real_tn, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred_tn, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        ids = np.asarray(product_ids)
        orden, inv = np.unique(ids, return_inverse=True)
        y_real = np.bincount(inv, weights=y_real, minlength=len(orden))
        y_pred = np.bincount(inv, weights=y_pred, minlength=len(orden))
    den = np.abs(y_real).sum()
    return float('nan') if den == 0 else float(np.abs(y_real - y_pred).sum() / den)


def espacio_hiper(trial, techo_arboles, regularizacion, semilla):
    base = {
        'objective': 'regression', 'metric': 'mae', 'verbosity': -1,
        'boosting_type': 'gbdt', 'seed': semilla, 'subsample_freq': 1, 'n_jobs': -1,
        'n_estimators': int(techo_arboles),
    }
    if regularizacion == 'fuerte':
        base.update({
            'num_leaves': trial.suggest_int('num_leaves', 8, 64),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample': trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:
        base.update({
            'num_leaves': trial.suggest_int('num_leaves', 20, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    return base


def entrenar(df_pd_sub, params, meses_tr, eval_meses=None, early_stopping_rounds=50):
    idx = df_pd_sub.index[df_pd_sub['periodo'].isin(meses_tr)]
    if len(idx) == 0:
        raise ValueError(f'Sin filas de entrenamiento para los meses {meses_tr}')
    X, y = df_pd_sub.loc[idx, FEATURES], df_pd_sub.loc[idx, 'y']
    modelo = lgb.LGBMRegressor(**params)
    fit_kwargs = dict(categorical_feature=CAT_FEATURES)
    if eval_meses:
        idx_ev = df_pd_sub.index[df_pd_sub['periodo'].isin(eval_meses)]
        if len(idx_ev) == 0:
            raise ValueError(f'Sin filas de evaluacion para early stopping en {eval_meses}')
        fit_kwargs['eval_set'] = [(df_pd_sub.loc[idx_ev, FEATURES], df_pd_sub.loc[idx_ev, 'y'])]
        fit_kwargs['callbacks'] = [lgb.early_stopping(early_stopping_rounds, verbose=False)]
    modelo.fit(X, y, **fit_kwargs)
    del X, y
    gc.collect()
    return modelo


def evaluar_combo(modelo, df_pd_sub, meses_ev, kind):
    df_ev = df_pd_sub[df_pd_sub['periodo'].isin(meses_ev)]
    if len(df_ev) == 0:
        return float('nan'), None
    pred_y = modelo.predict(df_ev[FEATURES])
    pred_tn = reconstruir_target(pred_y, df_ev, kind)
    score = wape(df_ev['clase_tn'], pred_tn, df_ev['product_id'])
    return score, pred_tn


print("motor de entrenamiento listo")


## 7) Motor de la grilla — un combo completo (Optuna + reentreno + submit)

`correr_combo()` hace el ciclo de vida completo de UN combo (target x
escalado): prepara el target, busca hiperparametros con Optuna
(`storage` sqlite + `load_if_exists=True`, resume interno de trials),
reentrena val/test, reentrena FINAL con todo lo supervisado, predice sobre
`df_infer_universo`, mezcla con el fallback de los pares excluidos, arma el
CSV de submit contra la lista oficial completa, y guarda `resultado.json`.

**Resume a nivel combo**: si `resultado.json` ya existe para este combo, se
saltea todo el bloque de arriba (ni siquiera se reabre el study de Optuna) —
asi retomar la grilla despues de un corte no relanza combos ya terminados.


In [ ]:
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")

_cols_pd = sorted(set(FEATURES + COLS_ID + ['clase_tn', 'B0', 'B1', '_tn0_norm']))
_cats = [c for c in CAT_FEATURES if c in _cols_pd]


def _a_pandas(df_pl, cols):
    cols_ok = [c for c in cols if c in df_pl.columns]
    out = (df_pl.select(cols_ok)
                .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats if c in cols_ok])
                .to_pandas())
    for c in _cats:
        if c in out.columns:
            out[c] = out[c].astype('category')
    return out


def preparar_combo(target_name, metodo, df_train_base, df_infer_base):
    if target_name in PARAM['targets_escalado_nativos']:
        tr_out, kind = preparar_target_escalado(df_train_base, target_name, metodo)
        inf_out, _ = preparar_target_escalado(df_infer_base, target_name, metodo)
    elif target_name == 'delta_mean_12':
        tr_out, kind = preparar_target_delta_mean12(df_train_base)
        inf_out, _ = preparar_target_delta_mean12(df_infer_base)
    elif target_name == 'delta_reg':
        _ridge = fit_ridge_delta_reg(df_train_base.filter(pl.col('periodo').is_in(MESES_TRAIN)))
        tr_out, kind = preparar_target_delta_reg(df_train_base, _ridge)
        inf_out, _ = preparar_target_delta_reg(df_infer_base, _ridge)
    else:
        raise ValueError(target_name)
    return tr_out, inf_out, kind


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


def correr_combo(target_name, metodo):
    tag_esc = metodo or 'fijo'
    experimento = f"otropipe__{target_name}__{tag_esc}"
    dir_out = RUTA_EXP / experimento

    if (dir_out / 'resultado.json').exists():
        print(f"[{experimento}] ya tiene resultado.json -> RESUME (se saltea)")
        return _leer_json_reintentando(dir_out / 'resultado.json')

    dir_out.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*70}\n{experimento}\n{'='*70}")

    tr_pl, inf_pl, kind = preparar_combo(target_name, metodo, df_sup_universo, df_infer_universo)
    df_pd = _a_pandas(tr_pl, _cols_pd + ['y'])
    df_infer_pd = _a_pandas(inf_pl, _cols_pd)

    db_local = Path.home() / f"optuna_{experimento}.db"
    db_bucket = RUTA_EXP / "optuna_db" / f"{experimento}.db"
    db_bucket.parent.mkdir(parents=True, exist_ok=True)
    if db_bucket.exists() and not db_local.exists():
        shutil.copy(db_bucket, db_local)
    storage = f"sqlite:///{db_local}"

    def objective(trial):
        params = espacio_hiper(trial, PARAM['techo_arboles'], PARAM['regularizacion'], PARAM['semilla'])
        modelo = entrenar(df_pd, params, MESES_TRAIN, eval_meses=MESES_VAL,
                          early_stopping_rounds=PARAM['early_stopping_rounds'])
        score, _ = evaluar_combo(modelo, df_pd, MESES_VAL, kind)
        if np.isnan(score):
            raise optuna.TrialPruned()
        bi = getattr(modelo, 'best_iteration_', None)
        trial.set_user_attr('best_iteration', int(bi) if bi else int(params['n_estimators']))
        return float(score)

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                study_name=experimento, storage=storage, load_if_exists=True)
    print(f"  trials previos: {len(study.trials)}  ->  corriendo {PARAM['n_trials']} nuevos")

    def respaldar():
        try:
            tmp = db_bucket.with_suffix('.db.tmp')
            shutil.copy(db_local, tmp)
            tmp.replace(db_bucket)
        except Exception as e:
            print(f"    [aviso] no se pudo respaldar: {e}")

    cada = max(1, int(PARAM['backup_cada_n_trials']))
    with tqdm(total=PARAM['n_trials'], desc=experimento[:40]) as pbar:
        def cb(st, tr):
            pbar.update(1)
            try:
                pbar.set_postfix({'mejor WAPE': f"{st.best_value:.5f}"})
            except ValueError:
                pass
            if pbar.n % cada == 0:
                respaldar()
        study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[cb])
    respaldar()
    print(f"  mejor WAPE (val): {study.best_value:.5f}")

    mejores = espacio_hiper(optuna.trial.FixedTrial(study.best_params),
                            PARAM['techo_arboles'], PARAM['regularizacion'], PARAM['semilla'])
    n_arboles_final = int(study.best_trial.user_attrs.get('best_iteration') or PARAM['techo_arboles'])
    mejores_fijos = dict(mejores); mejores_fijos['n_estimators'] = n_arboles_final

    modelo_val = entrenar(df_pd, mejores_fijos, MESES_TRAIN)
    wape_val, _ = evaluar_combo(modelo_val, df_pd, MESES_VAL, kind)

    meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
    modelo_test = entrenar(df_pd, mejores_fijos, meses_fit_test)
    wape_test, _ = evaluar_combo(modelo_test, df_pd, MESES_TEST, kind)
    print(f"  VAL wape={wape_val:.5f}   TEST wape={wape_test:.5f}")

    def _wape_naive(meses):
        _n = df_pd[df_pd['periodo'].isin(meses)]
        return wape(_n['clase_tn'], _n['tn0'] if 'tn0' in _n.columns else _n['clase_tn'] * 0, _n['product_id']) \
              if len(_n) else float('nan')
    naive_val, naive_test = _wape_naive(MESES_VAL), _wape_naive(MESES_TEST)

    meses_fit_final = sorted(set(df_pd['periodo'].unique().tolist()))
    modelo_final = entrenar(df_pd, mejores_fijos, meses_fit_final)

    pred_infer_y = modelo_final.predict(df_infer_pd[FEATURES])
    tn_pred_universo = reconstruir_target(pred_infer_y, df_infer_pd, kind)
    pred_infer = df_infer_pd[['product_id', 'customer_id', 'periodo', 'periodo_objetivo']].copy()
    pred_infer['tn_pred'] = np.maximum(tn_pred_universo, PARAM['clip_min'])

    OBJ = PARAM['periodo_objetivo']
    combinado = pl.concat([
        pl.from_pandas(pred_infer).select('product_id', 'periodo_objetivo',
                                          pl.col('tn_pred').alias('tn')),
        FALLBACK_INFER.select('product_id', 'periodo_objetivo', pl.col('tn_pred_fallback').alias('tn')),
    ], how='vertical_relaxed')
    obj = combinado.filter(pl.col('periodo_objetivo') == OBJ)
    if obj.is_empty():
        raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                           f"{sorted(combinado['periodo_objetivo'].unique().to_list())}")
    por_producto = obj.group_by('product_id').agg(pl.col('tn').sum().alias('tn'))
    submit = (oficiales.select('product_id').join(por_producto, on='product_id', how='left'))
    sin_pred = int(submit['tn'].null_count())
    submit = submit.with_columns(pl.col('tn').fill_null(0.0)).sort('product_id')

    path_submit = dir_out / f"submission_{OBJ}.csv"
    submit.write_csv(path_submit)
    print(f"  submit: {submit.height} filas, {sin_pred} sin prediccion, tn total {submit['tn'].sum():,.1f}")

    resultado = {
        'experimento': experimento, 'target': target_name, 'escalado': metodo, 'kind': kind,
        'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
        'periodo_objetivo': OBJ,
        'wape_val': wape_val, 'wape_test': wape_test,
        'wape_naive_val': naive_val, 'wape_naive_test': naive_test,
        'n_trials_total': len(study.trials), 'n_estimators_final': n_arboles_final,
        'hiperparametros': study.best_params,
        'n_filas_universo': int(tr_pl.height), 'n_pares_universo': int(UNIVERSO.height),
        'sin_prediccion': sin_pred, 'tn_total_entregado': float(submit['tn'].sum()),
        'path_submit': str(path_submit.relative_to(BUCKET)),
        'semilla': PARAM['semilla'],
    }
    with open(dir_out / 'resultado.json', 'w', encoding='utf-8') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
    shutil.copy(db_local, db_bucket)
    print(f"  guardado en {dir_out.relative_to(BUCKET)}")
    return resultado


print("correr_combo() listo")


## 8) Correr la grilla completa (resumible)

Cada combo que ya tiene `resultado.json` se saltea (resume). El
`manifest.json` a nivel grilla queda para saber, de un vistazo, que combos
estan listos sin tener que abrir cada `resultado.json`.


In [ ]:
t0 = time.time()
resultados = []
for target_name, metodo in GRILLA:
    r = correr_combo(target_name, metodo)
    resultados.append(r)

manifest = {
    'grilla': [{'target': t, 'escalado': e} for t, e in GRILLA],
    'n_combos': len(GRILLA),
    'resultados': [
        str((RUTA_EXP / f"otropipe__{r['target']}__{r['escalado'] or 'fijo'}" / 'resultado.json')
           .relative_to(BUCKET))
        for r in resultados
    ],
}
with open(RUTA_EXP / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)
print(f"\nGrilla completa: {len(resultados)}/{len(GRILLA)} combos.   [{time.time()-t0:.0f}s]")


## 9) Submit — TODOS los combos que terminaron, a Kaggle, pase lo que pase


In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. Los CSV ya estan generados.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                print(f"Kaggle auth copiada de {cand}")
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube nada. Los CSV ya estan generados.")
    else:
        kdst.chmod(0o600)
        print(f"Subiendo {len(resultados)} submits a Kaggle (sin filtrar por cuota)...")
        for i, r in enumerate(resultados, 1):
            path_submit = BUCKET / r['path_submit']
            msg = f"{r['experimento']} | wape_test={r['wape_test']:.5f}"
            ok, salida = kaggle_cli(["competitions", "submit", "-c", PARAM['kaggle_competition'],
                                    "-f", str(path_submit), "-m", msg])
            print(f"  [{i}/{len(resultados)}] {r['experimento']}: " + ("OK" if ok else "FALLO"))
            if not ok:
                print(f"    {salida[-300:]}")
            time.sleep(PARAM['pausa_entre_submits_seg'])


## 10) Leaderboard


In [ ]:
filas = []
for r in resultados:
    _nt = r.get('wape_naive_test')
    filas.append({
        'experimento': r['experimento'], 'target': r['target'], 'escalado': r['escalado'],
        'wape_val': round(r['wape_val'], 5), 'wape_test': round(r['wape_test'], 5),
        'wape_naive_test': round(_nt, 5) if _nt == _nt else None,
        'brecha_test_val': round(r['wape_test'] - r['wape_val'], 5),
        'sin_prediccion': r['sin_prediccion'], 'tn_entregado': round(r['tn_total_entregado'], 1),
    })
leaderboard = pl.DataFrame(filas).sort('wape_test')
leaderboard.write_csv(RUTA_EXP / 'leaderboard_otro_pipe.csv')
print(f"leaderboard_otro_pipe.csv ({leaderboard.height} combos):")
print(leaderboard)
